In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 04_transform_silver_transactions
# Objectif : Nettoyer et fiabiliser les transactions
#            (Bronze -> Silver) en mode APPEND-ONLY avec
#            contrôle d'idempotence (anti-join)
# Domaine  : Banking Lakehouse
# =====================================================

from pyspark.sql.functions import (
    col, current_timestamp, sha2, concat_ws, lit, when, row_number
)
from pyspark.sql.window import Window

# --- Configuration ---
TABLE_TRANSACTIONS_BRONZE = f"{CATALOG}.bronze.transactions_raw"
TABLE_TRANSACTIONS_SILVER = f"{CATALOG}.silver.transactions"

print("✅ Configuration Silver Transactions chargée")
print(f"Source Bronze : {TABLE_TRANSACTIONS_BRONZE}")
print(f"Cible Silver  : {TABLE_TRANSACTIONS_SILVER}")

In [0]:
# =====================================================
# Génération d'une clé technique (transaction_id)
# Hash sur TOUTES les colonnes numériques pour garantir
# l'unicité (déterministe et idempotent)
# + Application des règles de Data Quality
# =====================================================

df_bronze_transactions = spark.table(TABLE_TRANSACTIONS_BRONZE)

print(f"📊 Nombre de lignes en Bronze : {df_bronze_transactions.count()}")

# Liste de toutes les colonnes V1 à V28 + Time + Amount pour un hash robuste
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]

df_transactions_quality = (
    df_bronze_transactions
    .withColumn(
        "transaction_id",
        sha2(concat_ws("||", *[col(c) for c in feature_cols]), 256)
    )
    .withColumn(
        "dq_valid_amount",
        when(col("Amount") >= 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_time",
        when(col("Time") >= 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_class",
        when(col("Class").isin([0, 1]), lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_is_valid",
        col("dq_valid_amount") & col("dq_valid_time") & col("dq_valid_class")
    )
    .withColumn("_silver_processed_at", current_timestamp())
)

# --- Rapport de qualité ---
nb_total = df_transactions_quality.count()
nb_valid = df_transactions_quality.filter(col("dq_is_valid") == True).count()
nb_invalid = nb_total - nb_valid

print(f"\n📋 Rapport de Data Quality :")
print(f"📊 Total lignes         : {nb_total}")
print(f"✅ Lignes valides        : {nb_valid} ({round(100*nb_valid/nb_total, 4)}%)")
print(f"⚠️  Lignes avec anomalie : {nb_invalid} ({round(100*nb_invalid/nb_total, 4)}%)")

nb_ids_distincts = df_transactions_quality.select("transaction_id").distinct().count()
print(f"\n🔎 Nombre de transaction_id distincts : {nb_ids_distincts} / {nb_total}")

display(df_transactions_quality.select(
    "transaction_id", "Time", "Amount", "Class", "dq_is_valid"
).limit(5))

In [0]:
# =====================================================
# Déduplication des doublons EXACTS détectés en Bronze
# (les 1081 doublons originaux découverts en Phase 3.A)
# =====================================================

window_dedup_tx = Window.partitionBy("transaction_id").orderBy(col("_ingestion_timestamp").asc())

df_transactions_deduped = (
    df_transactions_quality
    .withColumn("_row_num", row_number().over(window_dedup_tx))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

nb_avant = df_transactions_quality.count()
nb_apres = df_transactions_deduped.count()

print(f"📊 Lignes avant déduplication Bronze : {nb_avant}")
print(f"📊 Lignes après déduplication Bronze  : {nb_apres}")
print(f"🗑️  Doublons exacts supprimés          : {nb_avant - nb_apres}")

nb_ids_finaux = df_transactions_deduped.select("transaction_id").distinct().count()
print(f"\n✅ Unicité finale : {nb_ids_finaux} IDs distincts / {df_transactions_deduped.count()} lignes")

In [0]:
# =====================================================
# Écriture en Silver - Version AVEC IDEMPOTENCE
# Empêche la ré-ingestion de transaction_id déjà existants
# (Correction du bug de duplication x5 découvert via monitoring)
# =====================================================

df_final_transactions = df_transactions_deduped.filter(col("dq_is_valid") == True)

table_exists = spark.catalog.tableExists(TABLE_TRANSACTIONS_SILVER)

if not table_exists:
    print(f"🆕 Table {TABLE_TRANSACTIONS_SILVER} n'existe pas -> création initiale")
    df_final_transactions.write.format("delta").saveAsTable(TABLE_TRANSACTIONS_SILVER)
else:
    print(f"🔍 Table {TABLE_TRANSACTIONS_SILVER} existe -> vérification idempotence")
    
    df_existing_ids = spark.table(TABLE_TRANSACTIONS_SILVER).select("transaction_id")
    
    df_new_only = df_final_transactions.join(
        df_existing_ids,
        on="transaction_id",
        how="left_anti"
    )
    
    nb_new = df_new_only.count()
    nb_skipped = df_final_transactions.count() - nb_new
    
    print(f"📊 Nouvelles transactions à ajouter : {nb_new}")
    print(f"⏭️  Transactions déjà présentes (ignorées) : {nb_skipped}")
    
    if nb_new > 0:
        df_new_only.write.format("delta").mode("append").saveAsTable(TABLE_TRANSACTIONS_SILVER)
        print(f"✅ {nb_new} nouvelle(s) transaction(s) ajoutée(s)")
    else:
        print("ℹ️ Aucune nouvelle transaction à ajouter (pipeline idempotent confirmé)")

nb_final = spark.table(TABLE_TRANSACTIONS_SILVER).count()
print(f"\n✅ Table Silver Transactions : {nb_final} lignes")

# --- Vérification répartition fraude en Silver ---
print("\n📈 Répartition Class en Silver (post-nettoyage) :")
spark.table(TABLE_TRANSACTIONS_SILVER).groupBy("Class").count().orderBy("Class").show()